# 00. Label Studio 환경 확인

이 노트북은 실제 Label Studio project나 task를 만들지 않고, 현재 Python 환경과 `.env` 설정이 준비되어 있는지만 확인합니다.

처음 사용하는 순서:

1. terminal에서 이 repo로 이동합니다.
2. `conda activate ltb_ultra`를 실행합니다.
3. `python -m pip install -e .`를 한 번 실행합니다.
4. `.env.example`을 `.env`로 복사하고 본인 서버 값으로 수정합니다.
5. 이 노트북을 같은 conda 환경 kernel로 실행합니다.

주의: API key 값은 출력하지 않습니다.

In [ ]:
from pathlib import Path
import os
import sys

# notebook 위치가 바뀌어도 repo root를 찾기 위한 보조 함수입니다.
def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise RuntimeError("repo root를 찾지 못했습니다. notebook을 repo 안에서 실행 중인지 확인하세요.")

REPO_ROOT = find_repo_root(Path.cwd())
print("REPO_ROOT =", REPO_ROOT)
print("Python =", sys.executable)

In [ ]:
# package가 editable install 되어 있으면 바로 import됩니다.
# 아직 설치하지 않았다면, 임시로 src를 sys.path에 추가해서 notebook 확인은 가능하게 합니다.
import sys
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import labelstudio_bbox_tools
print("labelstudio_bbox_tools version =", labelstudio_bbox_tools.__version__)

In [ ]:
from labelstudio_bbox_tools.config import load_dotenv

ENV_PATH = REPO_ROOT / ".env"
load_dotenv(ENV_PATH)

required = [
    "LABEL_STUDIO_URL",
    "LABEL_STUDIO_API_KEY",
    "LABEL_STUDIO_DOC_ROOT",
]

for name in required:
    value = os.environ.get(name)
    if name.endswith("API_KEY") and value:
        shown = "<set, hidden>"
    else:
        shown = value or "<missing>"
    print(f"{name} = {shown}")

In [ ]:
# 실제 Label Studio 연결까지 확인하고 싶을 때만 True로 바꾸세요.
# True로 바꿔도 project/task를 만들지는 않습니다.
CHECK_CONNECTION = False

if CHECK_CONNECTION:
    from labelstudio_bbox_tools.config import settings_from_env
    from labelstudio_bbox_tools.ls_client import make_client

    settings = settings_from_env(ENV_PATH)
    client = make_client(settings.url, settings.api_key)
    print("Label Studio connection OK:", settings.url)
else:
    print("CHECK_CONNECTION=False 이므로 서버 연결 확인은 건너뜁니다.")